# CREMA-D exploratory data analysis

This notebook inspects filename labels, audio metadata, class/speaker balance, acoustic examples, and speaker-independent splits. It reads `DATA_DIR` (default `/data/crema-d`) and does not modify source audio. No dataset is fabricated if the path is missing.

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from ser.data import scan_dataset, make_splits
DATA_DIR = os.environ.get('DATA_DIR', '/data/crema-d')
try:
    df, invalid = scan_dataset(DATA_DIR)
except FileNotFoundError as exc:
    print(f'Dataset unavailable: {exc}. Mount CREMA-D and rerun this cell.')
    df, invalid = pd.DataFrame(), pd.DataFrame()
print('Valid:', len(df), '| invalid/unparsed:', len(invalid))
display(invalid.head())

In [ ]:
if df.empty:
    print('No valid CREMA-D files found. Remaining data analysis is skipped.')
else:
    display(df.head())
    print('Speakers:', df.speaker_id.nunique(), 'Sample rates:', sorted(df.sample_rate.unique()), 'Channels:', sorted(df.channels.unique()))
    fig, ax = plt.subplots(1, 2, figsize=(13,4))
    df.emotion.value_counts().reindex(['angry','disgust','fear','happy','neutral','sad']).plot.bar(ax=ax[0],color='#8074c8'); ax[0].set(title='Emotion counts',xlabel='Emotion',ylabel='Audio files')
    df.groupby('speaker_id').size().plot.hist(bins=20,ax=ax[1],color='#52a99b'); ax[1].set(title='Files per speaker',xlabel='Audio files',ylabel='Speakers')
    plt.tight_layout()

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(11,5)); sns.countplot(data=df,x='speaker_id',hue='emotion',ax=ax); ax.tick_params(axis='x',rotation=90); ax.set(title='Speaker by emotion',xlabel='Speaker ID',ylabel='Audio files'); plt.tight_layout()

In [ ]:
if not df.empty:
    display(df.duration.describe())
    fig, ax = plt.subplots(figsize=(8,4)); sns.histplot(df.duration,bins=30,ax=ax,color='#e89a74'); ax.set(title='Audio duration distribution',xlabel='Duration (seconds)',ylabel='Files'); plt.show()
    display(df.groupby(['sample_rate','channels','subtype']).size().rename('files').reset_index())

In [ ]:
if not df.empty:
    import librosa, librosa.display
    examples=df.groupby('emotion',group_keys=False).head(1)
    for _, row in examples.iterrows():
        y,sr=librosa.load(row.path,sr=16000,mono=True)
        fig,ax=plt.subplots(1,3,figsize=(15,3))
        t=np.arange(len(y))/sr; ax[0].plot(t,y,lw=.5); ax[0].set(title=f"{row.emotion} waveform",xlabel='Time (s)',ylabel='Amplitude')
        D=librosa.amplitude_to_db(np.abs(librosa.stft(y)),ref=np.max); librosa.display.specshow(D,sr=sr,x_axis='time',y_axis='hz',ax=ax[1],cmap='magma'); ax[1].set(title='STFT magnitude (dB)')
        M=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=64); librosa.display.specshow(librosa.power_to_db(M,ref=np.max),sr=sr,x_axis='time',y_axis='mel',ax=ax[2],cmap='magma'); ax[2].set(title='Log-Mel (dB)')
        plt.tight_layout(); plt.show()

In [ ]:
if not df.empty:
    y,sr=librosa.load(df.iloc[0].path,sr=16000,mono=True)
    frame=librosa.feature.rms(y=y)[0]
    silence_fraction=float(np.mean(frame < 0.01))
    print(f'Example RMS silence fraction (<0.01): {silence_fraction:.1%}. This threshold is a rough diagnostic, not a universal silence definition.')
    splits=make_splits(df,seed=42); display(splits.groupby(['split','emotion']).size().unstack(fill_value=0))
    speaker_sets={s:set(splits.loc[splits.split==s,'speaker_id']) for s in ['train','validation','test']}
    print('Speaker overlaps:', {f'{a}-{b}': sorted(speaker_sets[a]&speaker_sets[b]) for a,b in [('train','validation'),('train','test'),('validation','test')]})
    display(splits.groupby('split').agg(files=('path','size'),speakers=('speaker_id','nunique')))
    splits.to_csv('../outputs/eda_split_manifest.csv',index=False)

## Interpretation checklist
Review class balance, speaker balance, duration and recording metadata above. Report malformed inputs from the `invalid` table. The group split protects speaker independence; exact class proportions can differ because speakers, rather than individual clips, are assigned as groups. CREMA-D is acted speech from a limited speaker pool, so performance may not transfer to spontaneous emotion or unseen recording conditions.